# Macro Regime & Market Risk Monitor
## 02 · Quarterly Cross-Asset Analysis

Combine Notebook 01's US/Eurozone regimes and rates with equity price changes,
EUR/USD price changes, and the Baa–10Y spread. The primary output is the
validated **36-column `cross_asset_quarterly.csv`**, ready for the existing SQL workflow.

Each regional market stress count has **three channels**:

- **US:** negative S&P 500 return, Baa spread QoQ change **above 0.05 percentage points**, and negative 10Y–2Y spread.
- **Eurozone:** negative EURO STOXX 50 return, negative 10Y–2Y spread, and the upstream **Widening** Italy–Germany fragmentation signal.

Multi-channel stress means **at least two** channels in either region. The
existing broad-stress summaries use **at least three**. Expected real 2Y rates
are reported as monetary conditions and do not enter either stress count.

Returns are percentage **price changes** from unadjusted Yahoo `Close`
(`auto_adjust=False`), rather than total-return series. Spread/rate changes
retain their source percentage-point units. These descriptive signals do not
establish causality or form a trading strategy.


## 1. Setup and reproducible inputs

Install the project's tested, pinned requirements before running this notebook;
this notebook does not install packages. Run Notebook 01 first to produce
`regional_regime.csv`, `us_regime.csv`, and `market_confirmation.csv`.

Set `MACRO_PROJECT_DIR` to the repository root. CSV inputs and exports use `data/processed`, or `MACRO_OUTPUT_DIR` when set.
`MACRO_RAW_DIR` defaults to `<project>/data/raw`. The default
`MACRO_DATA_MODE=cached` reads the saved raw snapshot without downloading.
Set `MACRO_DATA_MODE=live` explicitly to refresh the five Notebook 02 raw files;
live FRED access reads `FRED_API_KEY` from the environment or a Colab secret.
Keys are excluded from saved files, exceptions, and download metadata.

The original research cutoffs remain fixed: Yahoo starts on **2008-01-01**,
with exclusive end **2026-07-01**; quarterly analysis ends in **2026Q2**;
the IG/HY monthly exploratory appendix ends in **2026-08**.


In [ ]:
import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import yfinance as yf
from IPython.display import display

try:
    from google.colab import userdata
except ImportError:
    userdata = None

PROJECT_DIR = Path(os.environ.get("MACRO_PROJECT_DIR", Path.cwd())).resolve()
RAW_DIR = Path(os.environ.get("MACRO_RAW_DIR", PROJECT_DIR / "data" / "raw")).resolve()
OUTPUT_DIR = Path(os.environ.get("MACRO_OUTPUT_DIR", PROJECT_DIR / "data" / "processed")).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_MODE = os.environ.get("MACRO_DATA_MODE", "cached").lower()
if DATA_MODE not in {"cached", "live"}:
    raise ValueError("MACRO_DATA_MODE must be 'cached' or 'live'.")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
YAHOO_START = "2008-01-01"
YAHOO_END = "2026-07-01"  # Yahoo end dates are exclusive.
LAST_QUARTER = pd.Period("2026Q2", freq="Q")
LAST_CREDIT_MONTH = pd.Period("2026-08", freq="M")
JOIN_AUDIT = []
REGIME_COLUMNS = ["quarter", "us_regime", "eurozone_regime", "regime_divergence"]
US_REGIME_COLUMNS = ["quarter", "us_cpi_yoy", "real_gdp_growth_qoq", "regime"]
MARKET_COLUMNS = [
    "quarter", "us_10y_2y_spread", "eurozone_10y_2y_spread",
    "italy_germany_10y_spread", "us_expected_real_2y_rate",
    "eurozone_expected_real_2y_rate", "us_10y_yield", "eurozone_10y_yield",
    "us_10y_qoq_change", "eurozone_10y_qoq_change", "us_10y_direction",
    "eurozone_10y_direction", "fragmentation_signal",
]
print(f"Data mode: {DATA_MODE}; project: {PROJECT_DIR}; raw snapshot: {RAW_DIR}")


The small helpers below check keys and record join coverage. Inner joins retain
the original intersection of available quarters, while reporting unmatched
rows so that changing source coverage is visible. FRED missing-value markers
become missing numeric observations; daily market-calendar gaps are preserved.


In [ ]:
def require_columns(frame, columns, label, exact=False):
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise ValueError(f"{label}: missing columns {missing}")
    if exact and list(frame.columns) != list(columns):
        raise ValueError(f"{label}: unexpected column names or order")


def validate_key(frame, key, label):
    if frame.empty:
        raise ValueError(f"{label}: empty table")
    if frame[key].isna().any() or frame[key].duplicated().any():
        raise ValueError(f"{label}: missing or duplicate {key}")
    if not frame[key].is_monotonic_increasing:
        raise ValueError(f"{label}: {key} is out of order")


def require_nonmissing(frame, columns, label):
    null_counts = frame[columns].isna().sum()
    bad = null_counts[null_counts > 0].to_dict()
    if bad:
        raise ValueError(f"{label}: missing required values {bad}")


def read_quarterly_input(filename, columns):
    frame = pd.read_csv(OUTPUT_DIR / filename)
    require_columns(frame, columns, filename)
    if not frame["quarter"].astype("string").str.fullmatch(r"\d{4}Q[1-4]").fillna(False).all():
        raise ValueError(f"{filename}: quarter keys must have YYYYQn format")
    frame["quarter"] = pd.PeriodIndex(frame["quarter"], freq="Q")
    validate_key(frame, "quarter", filename)
    return frame


def merge_quarters(left, right, label):
    validate_key(left, "quarter", f"{label} left")
    validate_key(right, "quarter", f"{label} right")
    shared = (set(left.columns) & set(right.columns)) - {"quarter"}
    if shared:
        raise ValueError(f"{label}: ambiguous repeated columns {sorted(shared)}")
    merged = left.merge(right, on="quarter", how="inner", validate="one_to_one", sort=True)
    validate_key(merged, "quarter", label)
    left_keys, right_keys = set(left["quarter"]), set(right["quarter"])
    JOIN_AUDIT.append({
        "join": label, "left_rows": len(left), "right_rows": len(right),
        "matched_quarters": len(merged),
        "left_unmatched": len(left_keys - right_keys),
        "right_unmatched": len(right_keys - left_keys),
        "first_matched": str(merged["quarter"].min()),
        "last_matched": str(merged["quarter"].max()),
    })
    return merged


def record_raw_download(path, source, parameters):
    manifest_path = RAW_DIR / "manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {"sources": {}}
    manifest["sources"][path.stem] = {
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
        "provider": source, "parameters": parameters, "file": path.name,
        "identifier": parameters.get("series_id", parameters.get("tickers")),
        "pandas_version": pd.__version__, "yfinance_version": yf.__version__,
    }
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")


def verify_raw_snapshot(path):
    manifest_path = RAW_DIR / "manifest.json"
    if not manifest_path.exists():
        raise ValueError("Missing raw snapshot manifest.json")
    record = json.loads(manifest_path.read_text(encoding="utf-8"))["sources"].get(path.stem)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if record is None or record.get("sha256") != digest:
        raise ValueError(f"Missing snapshot record or hash mismatch for {path.name}")


def get_fred_series(series_id, column_name):
    path = RAW_DIR / f"fred_{series_id}.csv"
    if DATA_MODE == "live":
        key = os.environ.get("FRED_API_KEY")
        if not key and userdata is not None:
            try:
                key = userdata.get("FRED_API_KEY")
            except Exception:
                key = None
        if not key:
            raise RuntimeError("Live FRED access requires FRED_API_KEY; cached mode needs no key.")
        try:
            response = requests.get(
                "https://api.stlouisfed.org/fred/series/observations",
                params={"series_id": series_id, "api_key": key, "file_type": "json"},
                timeout=60,
            )
        except requests.RequestException:
            raise RuntimeError(f"FRED network request failed for {series_id}; URL omitted.") from None
        if not response.ok:
            raise RuntimeError(f"FRED HTTP {response.status_code} for {series_id}; URL omitted.")
        try:
            raw = pd.DataFrame(response.json()["observations"])[["date", "value"]]
        except (KeyError, ValueError):
            raise RuntimeError(f"Unexpected FRED response for {series_id}.") from None
        RAW_DIR.mkdir(parents=True, exist_ok=True)
        raw.to_csv(path, index=False)
        record_raw_download(path, "FRED", {"series_id": series_id})
    if not path.exists():
        raise FileNotFoundError(f"Missing raw snapshot {path.name}; populate RAW_DIR or explicitly choose live mode.")
    verify_raw_snapshot(path)
    frame = pd.read_csv(path)
    require_columns(frame, ["date", "value"], path.name, exact=True)
    frame["date"] = pd.to_datetime(frame["date"], errors="raise")
    frame["value"] = pd.to_numeric(frame["value"].replace(".", np.nan), errors="raise")
    frame = frame.sort_values("date").reset_index(drop=True)
    validate_key(frame, "date", path.name)
    if not frame["value"].notna().any():
        raise ValueError(f"{path.name}: no numeric observations")
    return frame.rename(columns={"value": column_name})


def get_yahoo_close(filename, ticker_names):
    path = RAW_DIR / filename
    names = list(ticker_names.values())
    if DATA_MODE == "live":
        downloaded = yf.download(
            list(ticker_names), start=YAHOO_START, end=YAHOO_END,
            auto_adjust=False, progress=False,
        )
        if downloaded.empty:
            raise RuntimeError(f"Yahoo returned no prices for {filename}")
        close = downloaded["Close"].copy()
        if isinstance(close, pd.Series):
            close = close.to_frame(name=next(iter(ticker_names)))
        close = close.rename(columns=ticker_names)[names]
        raw = close.rename_axis("date").reset_index()
        RAW_DIR.mkdir(parents=True, exist_ok=True)
        raw.to_csv(path, index=False)
        record_raw_download(path, "Yahoo Finance", {
            "tickers": list(ticker_names), "start": YAHOO_START,
            "exclusive_end": YAHOO_END, "field": "Close", "auto_adjust": False,
        })
    if not path.exists():
        raise FileNotFoundError(f"Missing raw snapshot {path.name}; populate RAW_DIR or explicitly choose live mode.")
    verify_raw_snapshot(path)
    frame = pd.read_csv(path)
    require_columns(frame, ["date", *names], path.name, exact=True)
    frame["date"] = pd.to_datetime(frame["date"], errors="raise")
    frame[names] = frame[names].apply(pd.to_numeric, errors="raise")
    frame = frame.sort_values("date").reset_index(drop=True)
    validate_key(frame, "date", path.name)
    if not frame["date"].between(pd.Timestamp(YAHOO_START), pd.Timestamp(YAHOO_END), inclusive="left").all():
        raise ValueError(f"{path.name}: dates outside the documented Yahoo window")
    for name in names:
        values = frame[name].dropna()
        if len(values) < 2 or not np.isfinite(values).all() or not (values > 0).all():
            raise ValueError(f"{path.name}: invalid {name} price observations")
    return frame.set_index("date").rename_axis("Date")


## 2. Validate Notebook 01 inputs

The upstream CSVs may contain additional exploratory columns. Only the named
contracts below enter the final table. Quarter keys must be unique and ordered.
Rate levels and expected real rates must be present. A first-period 10Y yield
change may be missing because its source has no earlier observation; later
missing changes fail validation. No value is filled to force a stress category.


In [ ]:
regional_regime = read_quarterly_input("regional_regime.csv", REGIME_COLUMNS)
us_regime = read_quarterly_input("us_regime.csv", US_REGIME_COLUMNS)
market_confirmation = read_quarterly_input("market_confirmation.csv", MARKET_COLUMNS)
require_nonmissing(regional_regime, REGIME_COLUMNS, "regional_regime")
require_nonmissing(us_regime, US_REGIME_COLUMNS, "us_regime")
change_columns = ["us_10y_qoq_change", "eurozone_10y_qoq_change"]
require_nonmissing(
    market_confirmation,
    [column for column in MARKET_COLUMNS if column not in change_columns],
    "market_confirmation",
)
first_market_quarter = market_confirmation["quarter"].min()
require_nonmissing(
    market_confirmation.loc[market_confirmation["quarter"] > first_market_quarter],
    change_columns, "market_confirmation after the first source quarter",
)
allowed_regimes = {"Goldilocks", "Overheating", "Stagflation", "Deflationary Slowdown"}
if not regional_regime[["us_regime", "eurozone_regime"]].isin(allowed_regimes).all().all():
    raise ValueError("Unexpected regional regime category")
if not us_regime["regime"].isin(allowed_regimes).all():
    raise ValueError("Unexpected US regime category")
if not pd.api.types.is_bool_dtype(regional_regime["regime_divergence"]):
    raise ValueError("regime_divergence must be Boolean")
expected_divergence = regional_regime["us_regime"] != regional_regime["eurozone_regime"]
if not regional_regime["regime_divergence"].equals(expected_divergence):
    raise ValueError("regime_divergence does not match the regional regime comparison")
display(pd.DataFrame([
    {"input": name, "rows": len(frame), "first_quarter": str(frame["quarter"].min()), "last_quarter": str(frame["quarter"].max())}
    for name, frame in [("regional_regime", regional_regime), ("us_regime", us_regime), ("market_confirmation", market_confirmation)]
]))


## 3. Equity price changes by regime

For each index, take the final available unadjusted daily close in each month
or quarter. Calculate the percentage change once. The first return lacks a
prior price and remains missing; missing prices are not forward-filled.


In [ ]:
equity_prices = get_yahoo_close(
    "yahoo_equity_close.csv", {"^GSPC": "sp500", "^STOXX50E": "eurostoxx50"}
)
equity_monthly = equity_prices.resample("ME").last().reset_index()
equity_monthly["month"] = equity_monthly["Date"].dt.to_period("M")
require_nonmissing(equity_monthly, ["sp500", "eurostoxx50"], "monthly equity closes")
equity_monthly["sp500_return"] = equity_monthly["sp500"].pct_change(fill_method=None) * 100
equity_monthly["eurostoxx50_return"] = equity_monthly["eurostoxx50"].pct_change(fill_method=None) * 100

equity_quarterly = equity_prices.resample("QE").last().reset_index()
equity_quarterly["quarter"] = equity_quarterly["Date"].dt.to_period("Q")
equity_quarterly = equity_quarterly[["quarter", "sp500", "eurostoxx50"]].copy()
require_nonmissing(equity_quarterly, ["sp500", "eurostoxx50"], "quarterly equity closes")
equity_quarterly["sp500_return"] = equity_quarterly["sp500"].pct_change(fill_method=None) * 100
equity_quarterly["eurostoxx50_return"] = equity_quarterly["eurostoxx50"].pct_change(fill_method=None) * 100
equity_quarterly = equity_quarterly.loc[equity_quarterly["quarter"] <= LAST_QUARTER].copy()
equity_regime_analysis = merge_quarters(
    regional_regime[REGIME_COLUMNS],
    equity_quarterly[["quarter", "sp500_return", "eurostoxx50_return"]],
    "regimes + equities",
)
display(equity_regime_analysis.tail(10))


In [ ]:
us_equity_by_regime = (
    equity_regime_analysis
    .groupby("us_regime")
    .agg(
        average_sp500_return=("sp500_return", "mean"),
        median_sp500_return=("sp500_return", "median"),
        observations=("sp500_return", "count")
    )
    .sort_values("average_sp500_return")
)

us_equity_by_regime


In [ ]:
eurozone_equity_by_regime = (
    equity_regime_analysis
    .groupby("eurozone_regime")
    .agg(
        average_eurostoxx_return=("eurostoxx50_return", "mean"),
        median_eurostoxx_return=("eurostoxx50_return", "median"),
        observations=("eurostoxx50_return", "count")
    )
    .sort_values("average_eurostoxx_return")
)

eurozone_equity_by_regime


In [ ]:
divergence_equity = (
    equity_regime_analysis
    .groupby("regime_divergence")
    .agg(
        average_sp500_return=("sp500_return", "mean"),
        average_eurostoxx_return=("eurostoxx50_return", "mean"),
        median_sp500_return=("sp500_return", "median"),
        median_eurostoxx_return=("eurostoxx50_return", "median"),
        observations=("sp500_return", "count")
    )
)

divergence_equity


## 4. Baa credit-spread analysis

The primary credit measure remains FRED **BAA10YM** (Baa yield minus 10Y
Treasury yield). Take its final available observation per quarter and compute
the level difference **before** joining regimes. A positive change above 0.05
percentage points is the existing US credit-stress criterion. IG/HY option-
adjusted spreads remain separate exploratory measures in the appendix.


In [ ]:
baa_spread = get_fred_series("BAA10YM", "us_baa_10y_spread")
baa_credit_quarterly = (
    baa_spread.assign(quarter=lambda frame: frame["date"].dt.to_period("Q"))
    .groupby("quarter").last().reset_index()
    [["quarter", "us_baa_10y_spread"]]
)
baa_credit_quarterly["baa_spread_qoq_change"] = baa_credit_quarterly["us_baa_10y_spread"].diff()
baa_credit_quarterly = baa_credit_quarterly.loc[baa_credit_quarterly["quarter"] <= LAST_QUARTER].copy()
baa_regime_analysis = merge_quarters(
    us_regime[US_REGIME_COLUMNS],
    baa_credit_quarterly[["quarter", "us_baa_10y_spread", "baa_spread_qoq_change"]],
    "US regime + Baa credit",
)
display(baa_regime_analysis.tail(10))


In [ ]:
baa_by_regime = (
    baa_regime_analysis
    .groupby("regime")
    .agg(
        quarters=("quarter", "count"),
        avg_spread=("us_baa_10y_spread", "mean"),
        median_spread=("us_baa_10y_spread", "median"),
        avg_spread_change=("baa_spread_qoq_change", "mean"),
        median_spread_change=("baa_spread_qoq_change", "median")
    )
    .reset_index()
)

baa_by_regime


In [ ]:
baa_widening_by_regime = (
    baa_regime_analysis
    .assign(
        baa_widened=baa_regime_analysis["baa_spread_qoq_change"] > 0
    )
    .groupby("regime")
    .agg(
        quarters=("quarter", "count"),
        widening_quarters=("baa_widened", "sum")
    )
    .reset_index()
)

baa_widening_by_regime["widening_pct"] = (
    baa_widening_by_regime["widening_quarters"]
    / baa_widening_by_regime["quarters"]
    * 100
)

baa_widening_by_regime


## 5. EUR/USD price changes and regional divergence

Use the final available daily EUR/USD close per quarter and its percentage
price change. Retain the original summaries by divergence and by the paired
regional regimes. These groups describe the contemporaneous dataset.


In [ ]:
eurusd_prices = get_yahoo_close("yahoo_eurusd_close.csv", {"EURUSD=X": "eurusd"})
eurusd_monthly = eurusd_prices.resample("ME").last().reset_index()
eurusd_monthly["month"] = eurusd_monthly["Date"].dt.to_period("M")
require_nonmissing(eurusd_monthly, ["eurusd"], "monthly EUR/USD closes")
eurusd_monthly["eurusd_return"] = eurusd_monthly["eurusd"].pct_change(fill_method=None) * 100
eurusd_quarterly = eurusd_prices.resample("QE").last().reset_index()
eurusd_quarterly["quarter"] = eurusd_quarterly["Date"].dt.to_period("Q")
require_nonmissing(eurusd_quarterly, ["eurusd"], "quarterly EUR/USD closes")
eurusd_quarterly["eurusd_return"] = eurusd_quarterly["eurusd"].pct_change(fill_method=None) * 100
eurusd_quarterly = eurusd_quarterly.loc[
    eurusd_quarterly["quarter"] <= LAST_QUARTER, ["quarter", "eurusd", "eurusd_return"]
].copy()
fx_regime_analysis = merge_quarters(
    regional_regime[REGIME_COLUMNS], eurusd_quarterly[["quarter", "eurusd_return"]],
    "regimes + EUR/USD",
)
display(fx_regime_analysis.tail(10))


In [ ]:
fx_divergence_summary = (
    fx_regime_analysis
    .groupby("regime_divergence")
    .agg(
        quarters=("eurusd_return", "count"),
        avg_eurusd_return=("eurusd_return", "mean"),
        median_eurusd_return=("eurusd_return", "median")
    )
    .reset_index()
)

fx_divergence_summary


In [ ]:
fx_regime_combinations = (
    fx_regime_analysis
    .groupby(["us_regime", "eurozone_regime"])
    .agg(
        quarters=("eurusd_return", "count"),
        avg_eurusd_return=("eurusd_return", "mean"),
        median_eurusd_return=("eurusd_return", "median")
    )
    .reset_index()
    .sort_values("avg_eurusd_return", ascending=False)
)

fx_regime_combinations


## 6. Build the quarterly cross-asset table

Join equities, FX, Baa credit, and Notebook 01's rate/fragmentation measures
once, using unique quarter keys. The rates retain the upstream expectation
sources and definitions. The coverage audit records every join intersection.


In [ ]:
cross_asset_prices = merge_quarters(
    equity_regime_analysis,
    fx_regime_analysis[["quarter", "eurusd_return"]], "equities + FX",
)
cross_asset_credit = merge_quarters(
    cross_asset_prices,
    baa_credit_quarterly[["quarter", "us_baa_10y_spread", "baa_spread_qoq_change"]],
    "cross-asset prices + Baa credit",
)
cross_asset_quarterly = merge_quarters(
    cross_asset_credit, market_confirmation[MARKET_COLUMNS],
    "cross-asset prices/credit + market confirmation",
)
if (cross_asset_quarterly["quarter"] > LAST_QUARTER).any():
    raise ValueError("Quarterly output extends beyond the documented cutoff")
display(pd.DataFrame(JOIN_AUDIT))
display(cross_asset_quarterly.tail(10))


In [ ]:
cross_asset_divergence_summary = (
    cross_asset_quarterly
    .groupby("regime_divergence")
    .agg(
        quarters=("quarter", "count"),
        avg_sp500_return=("sp500_return", "mean"),
        median_sp500_return=("sp500_return", "median"),
        avg_eurostoxx_return=("eurostoxx50_return", "mean"),
        median_eurostoxx_return=("eurostoxx50_return", "median"),
        avg_eurusd_return=("eurusd_return", "mean"),
        median_eurusd_return=("eurusd_return", "median")
    )
    .reset_index()
)

cross_asset_divergence_summary


In [ ]:
baa_divergence_summary = (
    cross_asset_quarterly
    .groupby("regime_divergence")
    .agg(
        quarters=("quarter", "count"),
        avg_credit_spread=("us_baa_10y_spread", "mean"),
        median_credit_spread=("us_baa_10y_spread", "median"),
        avg_credit_change=("baa_spread_qoq_change", "mean"),
        median_credit_change=("baa_spread_qoq_change", "median")
    )
    .reset_index()
)

baa_divergence_summary


## 7. Market signals and separate monetary conditions

Keep the corrected current notebook's categories and comparison boundaries.
Equity support/stress uses strictly positive/negative returns; a zero return
retains **Mixed Equity Signal**. Baa stress/improvement uses changes strictly
above **0.05** / below **−0.05**; boundary values remain **Credit Stable**.
Curve stress uses a strictly negative spread, so zero is not inverted.

A zero expected real rate retains **Positive** under the existing naming
convention. Expected real-rate signs remain monetary-condition indicators;
neither sign contributes to market stress counts.


In [ ]:
cross_asset_quarterly["equity_signal"] = "Mixed Equity Signal"

cross_asset_quarterly.loc[
    (cross_asset_quarterly["sp500_return"] > 0) &
    (cross_asset_quarterly["eurostoxx50_return"] > 0),
    "equity_signal"
] = "Equity Support"

cross_asset_quarterly.loc[
    (cross_asset_quarterly["sp500_return"] < 0) &
    (cross_asset_quarterly["eurostoxx50_return"] < 0),
    "equity_signal"
] = "Equity Stress"


In [ ]:
cross_asset_quarterly["credit_signal"] = "Credit Stable"

cross_asset_quarterly.loc[
    cross_asset_quarterly["baa_spread_qoq_change"] > 0.05,
    "credit_signal"
] = "Credit Stress"

cross_asset_quarterly.loc[
    cross_asset_quarterly["baa_spread_qoq_change"] < -0.05,
    "credit_signal"
] = "Credit Improvement"

cross_asset_quarterly["credit_signal"].value_counts()


In [ ]:
cross_asset_quarterly["curve_signal"] = "No Curve Stress"

cross_asset_quarterly.loc[
    (cross_asset_quarterly["us_10y_2y_spread"] < 0) &
    (cross_asset_quarterly["eurozone_10y_2y_spread"] < 0),
    "curve_signal"
] = "Both Curves Stress"

cross_asset_quarterly.loc[
    (
        (cross_asset_quarterly["us_10y_2y_spread"] < 0) &
        (cross_asset_quarterly["eurozone_10y_2y_spread"] >= 0)
    ) |
    (
        (cross_asset_quarterly["us_10y_2y_spread"] >= 0) &
        (cross_asset_quarterly["eurozone_10y_2y_spread"] < 0)
    ),
    "curve_signal"
] = "One Curve Stress"

cross_asset_quarterly["curve_signal"].value_counts()


In [ ]:
cross_asset_quarterly["forward_real_rate_signal"] = "Both Positive"

cross_asset_quarterly.loc[
    (cross_asset_quarterly["us_expected_real_2y_rate"] < 0) &
    (cross_asset_quarterly["eurozone_expected_real_2y_rate"] < 0),
    "forward_real_rate_signal"
] = "Both Negative"

cross_asset_quarterly.loc[
    (
        (cross_asset_quarterly["us_expected_real_2y_rate"] < 0) &
        (cross_asset_quarterly["eurozone_expected_real_2y_rate"] >= 0)
    ) |
    (
        (cross_asset_quarterly["us_expected_real_2y_rate"] >= 0) &
        (cross_asset_quarterly["eurozone_expected_real_2y_rate"] < 0)
    ),
    "forward_real_rate_signal"
] = "One Negative"

cross_asset_quarterly["forward_real_rate_signal"].value_counts()


In [ ]:
# -------------------------
# US market signals
# -------------------------

cross_asset_quarterly["us_equity_signal"] = "Mixed Equity Signal"

cross_asset_quarterly.loc[
    cross_asset_quarterly["sp500_return"] > 0,
    "us_equity_signal"
] = "Equity Support"

cross_asset_quarterly.loc[
    cross_asset_quarterly["sp500_return"] < 0,
    "us_equity_signal"
] = "Equity Stress"


cross_asset_quarterly["us_credit_signal"] = "Credit Stable"

cross_asset_quarterly.loc[
    cross_asset_quarterly["baa_spread_qoq_change"] > 0.05,
    "us_credit_signal"
] = "Credit Stress"

cross_asset_quarterly.loc[
    cross_asset_quarterly["baa_spread_qoq_change"] < -0.05,
    "us_credit_signal"
] = "Credit Improvement"


cross_asset_quarterly["us_curve_signal"] = "No Curve Stress"

cross_asset_quarterly.loc[
    cross_asset_quarterly["us_10y_2y_spread"] < 0,
    "us_curve_signal"
] = "Curve Stress"


cross_asset_quarterly["us_real_rate_signal"] = "Positive"

cross_asset_quarterly.loc[
    cross_asset_quarterly["us_expected_real_2y_rate"] < 0,
    "us_real_rate_signal"
] = "Negative"


In [ ]:
# -------------------------
# Eurozone market signals
# -------------------------

cross_asset_quarterly["eurozone_equity_signal"] = "Mixed Equity Signal"

cross_asset_quarterly.loc[
    cross_asset_quarterly["eurostoxx50_return"] > 0,
    "eurozone_equity_signal"
] = "Equity Support"

cross_asset_quarterly.loc[
    cross_asset_quarterly["eurostoxx50_return"] < 0,
    "eurozone_equity_signal"
] = "Equity Stress"


cross_asset_quarterly["eurozone_curve_signal"] = "No Curve Stress"

cross_asset_quarterly.loc[
    cross_asset_quarterly["eurozone_10y_2y_spread"] < 0,
    "eurozone_curve_signal"
] = "Curve Stress"


cross_asset_quarterly["eurozone_real_rate_signal"] = "Positive"

cross_asset_quarterly.loc[
    cross_asset_quarterly["eurozone_expected_real_2y_rate"] < 0,
    "eurozone_real_rate_signal"
] = "Negative"


cross_asset_quarterly["eurozone_fragmentation_signal"] = (
    cross_asset_quarterly["fragmentation_signal"]
)


## 8. Three-channel regional stress summaries

Count only the three documented regional channels. US credit and Eurozone
fragmentation represent different spread measures; their numeric counts
should be interpreted alongside the underlying channel labels.


In [ ]:
cross_asset_quarterly["us_market_stress_count"] = (
    (cross_asset_quarterly["us_equity_signal"] == "Equity Stress").astype(int)
    + (cross_asset_quarterly["us_credit_signal"] == "Credit Stress").astype(int)
    + (cross_asset_quarterly["us_curve_signal"] == "Curve Stress").astype(int)
)

cross_asset_quarterly["eurozone_market_stress_count"] = (
    (cross_asset_quarterly["eurozone_equity_signal"] == "Equity Stress").astype(int)
    + (cross_asset_quarterly["eurozone_curve_signal"] == "Curve Stress").astype(int)
    + (cross_asset_quarterly["eurozone_fragmentation_signal"] == "Widening").astype(int)
)

cross_asset_quarterly[
    [
        "quarter",
        "us_regime",
        "eurozone_regime",
        "us_market_stress_count",
        "eurozone_market_stress_count"
    ]
].tail(10)


In [ ]:
us_stress_by_regime = (
    cross_asset_quarterly
    .groupby("us_regime")
    .agg(
        quarters=("quarter", "count"),
        avg_stress_count=("us_market_stress_count", "mean"),
        multi_channel_quarters=(
            "us_market_stress_count",
            lambda x: (x >= 2).sum()
        ),
        broad_stress_quarters=(
            "us_market_stress_count",
            lambda x: (x >= 3).sum()
        )
    )
    .reset_index()
)

us_stress_by_regime["multi_channel_pct"] = (
    us_stress_by_regime["multi_channel_quarters"]
    / us_stress_by_regime["quarters"]
    * 100
)

us_stress_by_regime["broad_stress_pct"] = (
    us_stress_by_regime["broad_stress_quarters"]
    / us_stress_by_regime["quarters"]
    * 100
)

us_stress_by_regime


In [ ]:
eurozone_stress_by_regime = (
    cross_asset_quarterly
    .groupby("eurozone_regime")
    .agg(
        quarters=("quarter", "count"),
        avg_stress_count=("eurozone_market_stress_count", "mean"),
        multi_channel_quarters=(
            "eurozone_market_stress_count",
            lambda x: (x >= 2).sum()
        ),
        broad_stress_quarters=(
            "eurozone_market_stress_count",
            lambda x: (x >= 3).sum()
        )
    )
    .reset_index()
)

eurozone_stress_by_regime["multi_channel_pct"] = (
    eurozone_stress_by_regime["multi_channel_quarters"]
    / eurozone_stress_by_regime["quarters"]
    * 100
)

eurozone_stress_by_regime["broad_stress_pct"] = (
    eurozone_stress_by_regime["broad_stress_quarters"]
    / eurozone_stress_by_regime["quarters"]
    * 100
)

eurozone_stress_by_regime


In [ ]:
cross_asset_quarterly["macro_market_configuration"] = "No Multi-Channel Stress"

cross_asset_quarterly.loc[
    (
        (cross_asset_quarterly["us_market_stress_count"] >= 2) |
        (cross_asset_quarterly["eurozone_market_stress_count"] >= 2)
    ),
    "macro_market_configuration"
] = "Multi-Channel Stress"


## 9. Validate and export the SQL input

Match all **36 column names, order, and logical types** in the SQL source
contract. Reject extra or missing columns, duplicate quarters, missing core
observations, nonfinite values, invalid categories, or altered stress counts.
The only permitted missing output values are initial-source 10Y QoQ changes;
their existing direction default is retained. No missing value is replaced.

Export the main table and the existing equity-regime analysis only after
validation. IG/HY exploration follows the export and cannot change its schema
or its primary Baa credit methodology.


In [ ]:
OUTPUT_CONTRACT = {'quarter': 'VARCHAR', 'us_regime': 'VARCHAR', 'eurozone_regime': 'VARCHAR', 'regime_divergence': 'BOOLEAN', 'sp500_return': 'DOUBLE', 'eurostoxx50_return': 'DOUBLE', 'eurusd_return': 'DOUBLE', 'us_baa_10y_spread': 'DOUBLE', 'baa_spread_qoq_change': 'DOUBLE', 'us_10y_2y_spread': 'DOUBLE', 'eurozone_10y_2y_spread': 'DOUBLE', 'italy_germany_10y_spread': 'DOUBLE', 'us_expected_real_2y_rate': 'DOUBLE', 'eurozone_expected_real_2y_rate': 'DOUBLE', 'us_10y_yield': 'DOUBLE', 'eurozone_10y_yield': 'DOUBLE', 'us_10y_qoq_change': 'DOUBLE', 'eurozone_10y_qoq_change': 'DOUBLE', 'us_10y_direction': 'VARCHAR', 'eurozone_10y_direction': 'VARCHAR', 'fragmentation_signal': 'VARCHAR', 'equity_signal': 'VARCHAR', 'credit_signal': 'VARCHAR', 'curve_signal': 'VARCHAR', 'forward_real_rate_signal': 'VARCHAR', 'us_equity_signal': 'VARCHAR', 'us_credit_signal': 'VARCHAR', 'us_curve_signal': 'VARCHAR', 'us_real_rate_signal': 'VARCHAR', 'eurozone_equity_signal': 'VARCHAR', 'eurozone_curve_signal': 'VARCHAR', 'eurozone_real_rate_signal': 'VARCHAR', 'eurozone_fragmentation_signal': 'VARCHAR', 'us_market_stress_count': 'BIGINT', 'eurozone_market_stress_count': 'BIGINT', 'macro_market_configuration': 'VARCHAR'}
OUTPUT_COLUMNS = list(OUTPUT_CONTRACT)


In [ ]:
if len(cross_asset_quarterly.columns) != len(OUTPUT_COLUMNS) or set(cross_asset_quarterly.columns) != set(OUTPUT_COLUMNS):
    raise ValueError("Final table does not match the 36-column SQL source contract")
cross_asset_quarterly = cross_asset_quarterly[OUTPUT_COLUMNS].copy()
validate_key(cross_asset_quarterly, "quarter", "cross_asset_quarterly")
require_nonmissing(
    cross_asset_quarterly,
    [column for column in OUTPUT_COLUMNS if column not in change_columns],
    "cross_asset_quarterly core observations",
)
require_nonmissing(
    cross_asset_quarterly.loc[cross_asset_quarterly["quarter"] != first_market_quarter],
    change_columns, "cross_asset_quarterly after the first source quarter",
)
for column, logical_type in OUTPUT_CONTRACT.items():
    values = cross_asset_quarterly[column]
    if logical_type == "DOUBLE":
        if not pd.api.types.is_numeric_dtype(values) or not np.isfinite(values.dropna()).all():
            raise ValueError(f"{column}: expected finite numeric values")
        cross_asset_quarterly[column] = values.astype("float64")
    elif logical_type == "BIGINT":
        if not pd.api.types.is_integer_dtype(values):
            raise ValueError(f"{column}: expected integer values")
        cross_asset_quarterly[column] = values.astype("int64")
    elif logical_type == "BOOLEAN" and not pd.api.types.is_bool_dtype(values):
        raise ValueError(f"{column}: expected Boolean values")
    elif logical_type == "VARCHAR" and column != "quarter":
        if not values.map(lambda value: isinstance(value, str)).all():
            raise ValueError(f"{column}: expected text values")

categories = {
    "us_regime": allowed_regimes, "eurozone_regime": allowed_regimes,
    "equity_signal": {"Mixed Equity Signal", "Equity Support", "Equity Stress"},
    "us_equity_signal": {"Mixed Equity Signal", "Equity Support", "Equity Stress"},
    "eurozone_equity_signal": {"Mixed Equity Signal", "Equity Support", "Equity Stress"},
    "credit_signal": {"Credit Stable", "Credit Stress", "Credit Improvement"},
    "us_credit_signal": {"Credit Stable", "Credit Stress", "Credit Improvement"},
    "curve_signal": {"No Curve Stress", "One Curve Stress", "Both Curves Stress"},
    "us_curve_signal": {"No Curve Stress", "Curve Stress"},
    "eurozone_curve_signal": {"No Curve Stress", "Curve Stress"},
    "forward_real_rate_signal": {"Both Positive", "One Negative", "Both Negative"},
    "us_real_rate_signal": {"Positive", "Negative"},
    "eurozone_real_rate_signal": {"Positive", "Negative"},
    "fragmentation_signal": {"Widening", "Narrowing", "Unchanged"},
    "eurozone_fragmentation_signal": {"Widening", "Narrowing", "Unchanged"},
    "us_10y_direction": {"Rising", "Falling", "Unchanged"},
    "eurozone_10y_direction": {"Rising", "Falling", "Unchanged"},
    "macro_market_configuration": {"No Multi-Channel Stress", "Multi-Channel Stress"},
}
for column, allowed in categories.items():
    if not cross_asset_quarterly[column].isin(allowed).all():
        raise ValueError(f"{column}: unexpected category")

expected_us_count = (
    (cross_asset_quarterly["us_equity_signal"] == "Equity Stress").astype(int)
    + (cross_asset_quarterly["us_credit_signal"] == "Credit Stress").astype(int)
    + (cross_asset_quarterly["us_curve_signal"] == "Curve Stress").astype(int)
)
expected_eurozone_count = (
    (cross_asset_quarterly["eurozone_equity_signal"] == "Equity Stress").astype(int)
    + (cross_asset_quarterly["eurozone_curve_signal"] == "Curve Stress").astype(int)
    + (cross_asset_quarterly["eurozone_fragmentation_signal"] == "Widening").astype(int)
)
if not cross_asset_quarterly["us_market_stress_count"].equals(expected_us_count):
    raise ValueError("US stress count does not match its three channels")
if not cross_asset_quarterly["eurozone_market_stress_count"].equals(expected_eurozone_count):
    raise ValueError("Eurozone stress count does not match its three channels")
stress_columns = ["us_market_stress_count", "eurozone_market_stress_count"]
if not cross_asset_quarterly[stress_columns].isin([0, 1, 2, 3]).all().all():
    raise ValueError("Stress counts must be integers from 0 to 3")
expected_configuration = pd.Series(
    np.where((expected_us_count >= 2) | (expected_eurozone_count >= 2), "Multi-Channel Stress", "No Multi-Channel Stress"),
    index=cross_asset_quarterly.index,
)
if not cross_asset_quarterly["macro_market_configuration"].equals(expected_configuration):
    raise ValueError("Stress configuration does not match the existing >=2 rule")
print("Export validation passed.")
display(cross_asset_quarterly[stress_columns].agg(["min", "max"]))
display(cross_asset_quarterly["macro_market_configuration"].value_counts())


In [ ]:
cross_asset_export = cross_asset_quarterly.copy()
cross_asset_export["quarter"] = cross_asset_export["quarter"].astype(str)
cross_asset_export.to_csv(OUTPUT_DIR / "cross_asset_quarterly.csv", index=False, lineterminator="\r\n")
equity_regime_analysis.to_csv(OUTPUT_DIR / "equity_regime_analysis.csv", index=False, lineterminator="\r\n")
print(f"Saved {len(cross_asset_export)} quarters and {len(cross_asset_export.columns)} columns to cross_asset_quarterly.csv")
print("Saved equity_regime_analysis.csv")


## Appendix · IG/HY exploratory credit analysis

Preserve the separate FRED **BAMLC0A0CM** investment-grade and **BAMLH0A0HYM2**
high-yield option-adjusted spread analysis. These are different instruments
from BAA10YM. They do not replace the primary Baa methodology, do not enter
the regional stress counts, and do not add columns to the exported table.

The original aggregation is retained: final available daily observations per
month; monthly spread differences; final available month per quarter;
quarter-on-quarter differences in quarterly levels. The monthly appendix
cutoff is 2026-08 and the quarterly cutoff is 2026Q2.


In [ ]:
us_ig_spread = get_fred_series("BAMLC0A0CM", "us_ig_credit_spread")
us_hy_spread = get_fred_series("BAMLH0A0HYM2", "us_hy_credit_spread")
ig_hy_credit_daily = us_ig_spread.merge(
    us_hy_spread, on="date", how="outer", validate="one_to_one"
).sort_values("date")
validate_key(ig_hy_credit_daily, "date", "IG/HY daily spreads")
ig_hy_credit_monthly = ig_hy_credit_daily.set_index("date").resample("ME").last().reset_index()
ig_hy_credit_monthly["month"] = ig_hy_credit_monthly["date"].dt.to_period("M")
ig_hy_credit_monthly["ig_spread_change"] = ig_hy_credit_monthly["us_ig_credit_spread"].diff()
ig_hy_credit_monthly["hy_spread_change"] = ig_hy_credit_monthly["us_hy_credit_spread"].diff()
ig_hy_credit_monthly = ig_hy_credit_monthly.loc[ig_hy_credit_monthly["month"] <= LAST_CREDIT_MONTH].copy()
ig_hy_credit_quarterly = (
    ig_hy_credit_monthly.sort_values("month")
    .assign(quarter=lambda frame: frame["month"].dt.asfreq("Q"))
    .groupby("quarter").last().reset_index()
    [["quarter", "us_ig_credit_spread", "us_hy_credit_spread", "ig_spread_change", "hy_spread_change"]]
)
ig_hy_credit_quarterly = ig_hy_credit_quarterly.loc[ig_hy_credit_quarterly["quarter"] <= LAST_QUARTER].copy()
ig_hy_credit_quarterly["ig_spread_qoq_change"] = ig_hy_credit_quarterly["us_ig_credit_spread"].diff()
ig_hy_credit_quarterly["hy_spread_qoq_change"] = ig_hy_credit_quarterly["us_hy_credit_spread"].diff()
ig_hy_columns = ["quarter", "us_ig_credit_spread", "us_hy_credit_spread", "ig_spread_qoq_change", "hy_spread_qoq_change"]
ig_hy_regime_analysis = merge_quarters(
    regional_regime[REGIME_COLUMNS], ig_hy_credit_quarterly[ig_hy_columns],
    "appendix regimes + IG/HY credit",
)
ig_hy_us_regime_analysis = merge_quarters(
    us_regime[US_REGIME_COLUMNS], ig_hy_credit_quarterly[ig_hy_columns],
    "appendix US regime + IG/HY credit",
)
display(ig_hy_credit_monthly.tail(10))
display(ig_hy_regime_analysis.tail(10))
print("IG/HY appendix observations:", len(ig_hy_us_regime_analysis))


In [ ]:
ig_hy_by_regime = (
    ig_hy_regime_analysis
    .groupby("us_regime")
    .agg(
        average_ig_spread=("us_ig_credit_spread", "mean"),
        median_ig_spread=("us_ig_credit_spread", "median"),
        average_hy_spread=("us_hy_credit_spread", "mean"),
        median_hy_spread=("us_hy_credit_spread", "median"),
        observations=("us_ig_credit_spread", "count")
    )
)

ig_hy_by_regime


The earlier notebooks' exploratory fiscal, long-end yield, and realized versus
expected real-rate analyses remain in Notebook 01. This notebook's final
quarterly table uses the stated market inputs and the corrected three-channel
definition only. Cached data support rerunning the same snapshot; a live
refresh may differ because providers can revise macro history or market prices.
